In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import matplotlib as mpl
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

CKPT_RESNET18 = "best_model.pth"
CKPT_RESNET34 = "resnet34_multiscale_best.pth"
CKPT_DINOV2   = "dinov2_best_f1.pth"     # or dinov2_best_iou.pth

THRESH_RESNET18 = 0.50
THRESH_RESNET34 = 0.50
THRESH_DINOV2   = 0.30   # optimal threshold from your best-F1 checkpoint search

DATA_ROOT  = "."
TEST_FILE  = "test.txt"

NUM_QUALITATIVE_SAMPLES = 8
RANDOM_SEED = 42

In [2]:
class PSCDTestDataset(Dataset):
    """
    Same loading/normalization as your training PSCDDataset (BGR, /255.0,
    no ImageNet normalization — matches all three of your notebooks exactly),
    but with a DETERMINISTIC CENTER CROP instead of a random one, so that:
      (1) results are reproducible across runs,
      (2) all three models see the identical crop for a given index — required
          for a fair qualitative side-by-side comparison.
    """
    def __init__(self, root, split_file, patch_size=512):
        self.root = root
        self.patch_size = patch_size
        with open(split_file) as f:
            self.ids = [line.strip() for line in f.readlines()]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_t0 = cv2.imread(os.path.join(self.root, "t0",   img_id + ".png"))
        img_t1 = cv2.imread(os.path.join(self.root, "t1",   img_id + ".png"))
        mask   = cv2.imread(os.path.join(self.root, "mask", img_id + ".png"), cv2.IMREAD_GRAYSCALE)

        h, w = mask.shape
        ps = self.patch_size
        x = max(0, (w - ps) // 2)   # center crop, not random
        y = max(0, (h - ps) // 2)

        img_t0 = img_t0[y:y+ps, x:x+ps]
        img_t1 = img_t1[y:y+ps, x:x+ps]
        mask   = mask[y:y+ps, x:x+ps]

        img_t0_t = torch.from_numpy(img_t0.transpose(2, 0, 1)).float() / 255.0
        img_t1_t = torch.from_numpy(img_t1.transpose(2, 0, 1)).float() / 255.0
        mask_t   = (torch.from_numpy(mask) > 0).long()

        return {
            "img_t0": img_t0_t, "img_t1": img_t1_t, "mask": mask_t,
            "img_t0_bgr": img_t0, "img_t1_bgr": img_t1,  # kept raw for display
        }


test_dataset = PSCDTestDataset(root=DATA_ROOT, split_file=TEST_FILE)
print(f"Test set size: {len(test_dataset)}")

Test set size: 116


In [3]:
import torchvision.models as models

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.conv1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x = self.conv1(x)
        f1 = self.layer1(self.maxpool(x))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f1, f2, f3, f4


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ResNet18DecoderWithHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.up3 = UpBlock(512 + 256, 256)
        self.up2 = UpBlock(256 + 128, 128)
        self.up1 = UpBlock(128 + 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, d1, d2, d3, d4):
        x = self.up3(d4, d3)
        x = self.up2(x, d2)
        x = self.up1(x, d1)
        return self.head(x)


class MultiScaleSiameseUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ResNet18Encoder()
        self.decoder = ResNet18DecoderWithHead()

    def forward(self, t0, t1):
        f1_0, f2_0, f3_0, f4_0 = self.encoder(t0)
        f1_1, f2_1, f3_1, f4_1 = self.encoder(t1)
        d1 = torch.abs(f1_0 - f1_1)
        d2 = torch.abs(f2_0 - f2_1)
        d3 = torch.abs(f3_0 - f3_1)
        d4 = torch.abs(f4_0 - f4_1)
        return self.decoder(d1, d2, d3, d4)

In [4]:
class ResNet34Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        self.conv1 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x):
        x = self.conv1(x)
        f1 = self.layer1(self.maxpool(x))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f1, f2, f3, f4


class ResNet34DecoderWithHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.up3 = UpBlock(512 + 256, 256)   # reuses UpBlock defined in Cell 3
        self.up2 = UpBlock(256 + 128, 128)
        self.up1 = UpBlock(128 + 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, d1, d2, d3, d4):
        x = self.up3(d4, d3)
        x = self.up2(x, d2)
        x = self.up1(x, d1)
        return self.head(x)


class MultiScaleSiameseUNet34(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ResNet34Encoder()
        self.decoder = ResNet34DecoderWithHead()

    def forward(self, t0, t1):
        f1_0, f2_0, f3_0, f4_0 = self.encoder(t0)
        f1_1, f2_1, f3_1, f4_1 = self.encoder(t1)
        d1 = torch.abs(f1_0 - f1_1)
        d2 = torch.abs(f2_0 - f2_1)
        d3 = torch.abs(f3_0 - f3_1)
        d4 = torch.abs(f4_0 - f4_1)
        return self.decoder(d1, d2, d3, d4)

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)


class DINOAdapter(nn.Module):
    def __init__(self, embed_dim=384):
        super().__init__()
        self.up1   = nn.Sequential(nn.Upsample(size=(128,128), mode='bilinear'), nn.Conv2d(embed_dim, 64,  1))
        self.up2   = nn.Sequential(nn.Upsample(size=(64, 64),  mode='bilinear'), nn.Conv2d(embed_dim, 128, 1))
        self.up3   = nn.Sequential(nn.Upsample(size=(32, 32),  mode='bilinear'), nn.Conv2d(embed_dim, 256, 1))
        self.down4 = nn.Sequential(nn.AdaptiveAvgPool2d((16,16)),                nn.Conv2d(embed_dim, 512, 1))

    def forward(self, features):
        f1, f2, f3, f4 = features
        return self.up1(f1), self.up2(f2), self.up3(f3), self.down4(f4)


class SiameseDINOUNet(nn.Module):
    def __init__(self, freeze_dino=True):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

        if freeze_dino:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.adapter    = DINOAdapter(embed_dim=384)
        self.up4        = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4       = ConvBlock(512, 256)
        self.up3        = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3       = ConvBlock(256, 128)
        self.up2        = nn.ConvTranspose2d(128, 64,  2, 2)
        self.dec2       = ConvBlock(128, 64)
        self.up_final   = nn.Upsample(scale_factor=4, mode='bilinear')
        self.final_conv = nn.Conv2d(64, 1, 1)

    def extract_features(self, x):
        x = F.interpolate(x, size=(518, 518), mode='bilinear', align_corners=False)
        with torch.no_grad():
            features = self.backbone.get_intermediate_layers(x, n=[2, 5, 8, 11])
        reshaped = [
            f.view(f.shape[0], 37, 37, 384).permute(0, 3, 1, 2).contiguous()
            for f in features
        ]
        return self.adapter(reshaped)

    def forward(self, img_t0, img_t1):
        f1_0, f2_0, f3_0, f4_0 = self.extract_features(img_t0)
        f1_1, f2_1, f3_1, f4_1 = self.extract_features(img_t1)
        d1 = torch.abs(f1_0 - f1_1)
        d2 = torch.abs(f2_0 - f2_1)
        d3 = torch.abs(f3_0 - f3_1)
        d4 = torch.abs(f4_0 - f4_1)
        x = self.dec4(torch.cat([self.up4(d4), d3], dim=1))
        x = self.dec3(torch.cat([self.up3(x),  d2], dim=1))
        x = self.dec2(torch.cat([self.up2(x),  d1], dim=1))
        return F.interpolate(
            self.final_conv(self.up_final(x)),
            size=(512, 512), mode='bilinear', align_corners=False
        )

In [6]:
model_r18  = MultiScaleSiameseUNet().cuda()
model_r34  = MultiScaleSiameseUNet34().cuda()
model_dino = SiameseDINOUNet(freeze_dino=True).cuda()   # <-- corrected class name

model_r18.load_state_dict(torch.load(CKPT_RESNET18))
model_r34.load_state_dict(torch.load(CKPT_RESNET34))
model_dino.load_state_dict(torch.load(CKPT_DINOV2))

for m in (model_r18, model_r34, model_dino):
    m.eval()

models = {
    "ResNet18": {"model": model_r18,  "threshold": THRESH_RESNET18},
    "ResNet34": {"model": model_r34,  "threshold": THRESH_RESNET34},
    "PSCDLNet": {"model": model_dino, "threshold": THRESH_DINOV2},
}
print("All three models loaded successfully.")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


All three models loaded successfully.


In [7]:
def compute_metrics(pred_bin: np.ndarray, gt: np.ndarray, eps=1e-6):
    pred_bin = pred_bin.astype(np.float32)
    gt = gt.astype(np.float32)
    tp = float((pred_bin * gt).sum())
    fp = float((pred_bin * (1 - gt)).sum())
    fn = float(((1 - pred_bin) * gt).sum())
    tn = float(((1 - pred_bin) * (1 - gt)).sum())

    iou = (tp + eps) / (tp + fp + fn + eps)
    f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    return {"iou": iou, "f1": f1, "precision": precision, "recall": recall, "accuracy": accuracy}

In [8]:
def evaluate_model(model, threshold, test_dataset, label=""):
    all_m = {"iou": [], "f1": [], "precision": [], "recall": [], "accuracy": []}
    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(len(test_dataset)), desc=f"Testing [{label}]"):
            sample = test_dataset[idx]
            img_t0 = sample["img_t0"].unsqueeze(0).cuda()
            img_t1 = sample["img_t1"].unsqueeze(0).cuda()
            gt_mask = sample["mask"].numpy().reshape(-1)

            pred = torch.sigmoid(model(img_t0, img_t1))
            pred_bin = (pred.squeeze().cpu().numpy().reshape(-1) > threshold)

            m = compute_metrics(pred_bin, gt_mask)
            for k in all_m:
                all_m[k].append(m[k])

    summary = {k: float(np.mean(v)) * 100 for k, v in all_m.items()}
    print(f"\n========== Test Set Results — {label} ==========")
    print(f"  Threshold : {threshold:.2f}")
    print(f"  IoU       : {summary['iou']:.2f}%")
    print(f"  F1 Score  : {summary['f1']:.2f}%")
    print(f"  Precision : {summary['precision']:.2f}%")
    print(f"  Recall    : {summary['recall']:.2f}%")
    print(f"  Accuracy  : {summary['accuracy']:.2f}%")
    print("=" * (26 + len(label)))
    return summary

In [9]:
results = {}
for name, cfg in models.items():
    results[name] = evaluate_model(cfg["model"], cfg["threshold"], test_dataset, label=name)

Testing [ResNet18]: 100%|██████████| 116/116 [00:24<00:00,  4.67it/s]



========== Test Set Results — ResNet18 ==========
  Threshold : 0.50
  IoU       : 66.16%
  F1 Score  : 78.15%
  Precision : 85.74%
  Recall    : 74.77%
  Accuracy  : 94.79%


Testing [ResNet34]: 100%|██████████| 116/116 [00:25<00:00,  4.63it/s]



========== Test Set Results — ResNet34 ==========
  Threshold : 0.50
  IoU       : 64.79%
  F1 Score  : 77.22%
  Precision : 75.99%
  Recall    : 82.02%
  Accuracy  : 93.67%


Testing [PSCDLNet]: 100%|██████████| 116/116 [00:26<00:00,  4.37it/s]


========== Test Set Results — PSCDLNet ==========
  Threshold : 0.30
  IoU       : 65.55%
  F1 Score  : 77.99%
  Precision : 79.42%
  Recall    : 79.69%
  Accuracy  : 94.22%


In [10]:
def generate_qualitative_figure(test_dataset, models_dict, num_samples=8,
                                 seed=42, save_path="qualitative_comparison.pdf"):
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "pdf.fonttype": 42, "ps.fonttype": 42,
    })

    random.seed(seed)
    indices = random.sample(range(len(test_dataset)), num_samples)

    col_titles = ["Input $t_0$", "Input $t_1$", "Ground Truth"] + list(models_dict.keys())
    n_cols = len(col_titles)
    n_rows = num_samples

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.1 * n_cols, 2.1 * n_rows), dpi=300)
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row_idx, idx in enumerate(indices):
        sample = test_dataset[idx]
        # BGR -> RGB for correct display colors
        img0_disp = cv2.cvtColor(sample["img_t0_bgr"], cv2.COLOR_BGR2RGB)
        img1_disp = cv2.cvtColor(sample["img_t1_bgr"], cv2.COLOR_BGR2RGB)
        gt = sample["mask"].numpy()

        axes[row_idx, 0].imshow(img0_disp)
        axes[row_idx, 1].imshow(img1_disp)
        axes[row_idx, 2].imshow(gt, cmap="gray")

        with torch.no_grad():
            for col_idx, (name, cfg) in enumerate(models_dict.items()):
                img_t0 = sample["img_t0"].unsqueeze(0).cuda()
                img_t1 = sample["img_t1"].unsqueeze(0).cuda()
                pred = torch.sigmoid(cfg["model"](img_t0, img_t1))
                pred_bin = (pred.squeeze().cpu().numpy() > cfg["threshold"])
                axes[row_idx, 3 + col_idx].imshow(pred_bin, cmap="gray")

        for c in range(n_cols):
            axes[row_idx, c].axis("off")

    for c, title in enumerate(col_titles):
        axes[0, c].set_title(title, fontsize=13, fontweight="bold", pad=8)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.savefig(save_path.replace(".pdf", ".png"), bbox_inches="tight")
    plt.close()
    print(f"Saved qualitative comparison figure to {save_path}")

In [11]:
generate_qualitative_figure(test_dataset, models, num_samples=NUM_QUALITATIVE_SAMPLES,
                             seed=RANDOM_SEED, save_path="qualitative_comparison.pdf")

Saved qualitative comparison figure to qualitative_comparison.pdf


In [13]:
FIXED_INDICES = [323
,210
,148
,44
,275]

print(f"Test set has {len(test_dataset)} samples.")
print(f"Using fixed indices: {FIXED_INDICES}")

Test set has 116 samples.
Using fixed indices: [323, 210, 148, 44, 275]


In [15]:
print(f"Test set has {len(test_dataset)} samples (valid indices: 0 to {len(test_dataset) - 1})")

Test set has 116 samples (valid indices: 0 to 115)


In [16]:
n_test = len(test_dataset)
print(f"Test set has {n_test} samples.")

# Evenly spread 5 fixed indices across the whole test set — guaranteed in range,
# and still deterministic/fixed (no randomness) every time you run this.
FIXED_INDICES = [int(n_test * f) for f in [0.05, 0.25, 0.50, 0.75, 0.95]]
FIXED_INDICES = [min(i, n_test - 1) for i in FIXED_INDICES]  # safety clamp

print(f"Using fixed indices: {FIXED_INDICES}")
for i in FIXED_INDICES:
    print(f"  idx {i} -> {test_dataset.ids[i]}")

Test set has 116 samples.
Using fixed indices: [5, 29, 58, 87, 110]
  idx 5 -> 00000346
  idx 29 -> 00000181
  idx 58 -> 00000462
  idx 87 -> 00000383
  idx 110 -> 00000247


In [17]:
generate_qualitative_figure_fixed(test_dataset, models, FIXED_INDICES,
                                   save_path="qualitative_comparison_fixed.pdf")

Saved fixed-sample qualitative comparison to qualitative_comparison_fixed.pdf


In [18]:
FIXED_INDICES = [5, 29, 58, 87, 110]  # keep, or replace individual entries after reviewing the render